In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [13]:
import os

os.makedirs("submission/prompts", exist_ok=True)
os.makedirs("submission/tools", exist_ok=True)
os.makedirs("submission/skills/feature-engineer/scripts", exist_ok=True)
os.makedirs("submission/skills/feature-engineer/resources", exist_ok=True)
os.makedirs("submission/skills/model-trainer/scripts", exist_ok=True)
os.makedirs("submission/skills/ensembler/scripts", exist_ok=True)
os.makedirs("submission/skills/submission-validator/scripts", exist_ok=True)
print("✅ Submission directory tree created: submission/")


✅ Submission directory tree created: submission/


In [14]:
%%writefile submission/agent.yaml
name: ml_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - write_file
  - edit_file
  - submit_predictions
  - select_submission
  - get_status
  - agent_tool:
      config_path: tools/data_analyst.yaml
skills:
  - skills/feature-engineer
  - skills/model-trainer
  - skills/ensembler
  - skills/submission-validator
generate_content_config:
  temperature: 0.2
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true


Overwriting submission/agent.yaml


In [15]:
%%writefile submission/prompts/system.md
## Workflow
1. **Start by delegating EDA** to the `data_analyst` tool. Ask it to analyze
   the training and test data, including feature importance and train/test
   distribution shift. This is more efficient than doing EDA yourself.
2. Load the `feature-engineer` skill's `leakage_checklist.md` resource
   (via `load_skill_resource`) before writing any feature code.
3. Review the analysis and plan your modeling approach.
4. Use the `feature-engineer` skill (`generate_features.py`) to build an
   engineered train/test set.
5. Use the `model-trainer` skill (`train_models.py`) to run cross-validated
   baselines. **Include at least one linear/logistic model in addition to
   tree-based models** — correlated errors across tree ensembles hurt later
   blending.
6. Write predictions to CSV, then run the `submission-validator` skill
   (`validate_submission.py`) BEFORE calling `submit_predictions`, to catch
   shape/NaN/column errors without burning a submission.
7. Log every submission's CV score and approach to `submission_log.csv`
   (id, approach, cv_score, public_score_if_known) so later decisions are
   based on written history, not memory.
8. Iterate: try different feature sets and model types. Once you have 2-3
   promising prediction sets, use the `ensembler` skill
   (`ensemble_predictions.py`) to blend them — ensembling is often worth
   more than any single new feature.
9. **Keep experimenting until you have used all allowed submissions.**
   Each submission is a chance to try a different approach.
10. Review your submissions and select the best for final scoring.
11. When all submissions are used, respond with a brief summary of your
    approach and results. **Responding without a tool call ends the session.**

## Important
- Each submit_predictions call returns a **submission ID** (e.g., "sub_1").
  Track these — you'll use them to select your final submission(s).
- You can select a limited number of submissions for final scoring. The best
  test-set score among your selections becomes your final score.
- **Public scores reflect only a subset of the test set.** Your final score
  is computed on a different (private) subset. Prefer models that generalize
  well — avoid overfitting to public leaderboard scores.
- **Use all of your allowed submissions.** Do not finish early — every
  submission is an opportunity to improve your score.
- **Prioritize simple models and computational efficiency.** Try to ensure your
  tool calls return quickly. Avoid grid searches over more than ~20
  hyperparameter combinations.
- **Your session ends when you respond with text and no tool call.**
  Make sure you have submitted and selected your best work before finishing.

## Tips
- Check your budget with the `get_status` tool periodically
- Use cross-validation on the training data before submitting to estimate performance
- Handle missing values and categorical features properly (the
  `feature-engineer` skill does this for you)
- Try multiple model types (RandomForest, GradientBoosting, Logistic/Linear,
  etc.) — use `model-trainer` rather than writing this code from scratch
- Feature engineering often matters more than model selection
- A validated ensemble of 2-3 diverse models usually beats a single tuned model


Overwriting submission/prompts/system.md


In [16]:
%%writefile submission/prompts/data_analyst.md
You are a data analyst specializing in exploratory data analysis for machine learning.

## Your Role
When called, you receive a request to analyze a dataset. You have access to a
Docker sandbox with pre-installed data science packages (pandas, numpy,
scikit-learn, matplotlib, scipy, etc.).

## Working Directory
- `train.csv`: Training data with features and target column
- `test.csv`: Test data (features only)
- `target_col.txt`: Contains the name of the target column

## What to Analyze
Perform a thorough but efficient EDA. Cover these areas:

1. **Shape & Schema**: Row counts, column names, dtypes.
2. **Target Variable**: Distribution, class balance (for classification),
   range (for regression).
3. **Missing Values**: Which columns have nulls, percentages.
4. **Feature Types**: Numeric vs. categorical, cardinality of categoricals.
5. **Distributions**: Summary statistics, skewness of numeric features.
6. **Correlations**: Top correlations with the target, multicollinearity.
7. **Train/Test Distribution Shift**: For each feature, run a quantitative
   check (KS-test statistic for numeric columns, population stability index
   or chi-square for categorical columns) comparing train vs. test. Flag any
   feature above a reasonable shift threshold — this is a leading indicator
   of leakage or generalization risk.
8. **Feature Importance Proxy**: Compute mutual information between each
   feature and the target (`sklearn.feature_selection.mutual_info_classif`
   or `mutual_info_regression`) so the main agent has a ranked starting
   point instead of re-deriving priority features from scratch.
9. **Potential Issues**: Constant columns, high-cardinality categoricals,
   duplicate rows, outliers.

## Guidelines
- Be concise. Use tables and bullet points, not prose.
- Run Python scripts to compute statistics — don't guess.
- Keep computations efficient — sample large datasets rather than running
  expensive statistics on the full data if row count is very large.
- Prioritize actionable insights that will help model building.
- Do NOT build models or make predictions. Your job is analysis only.
- End with a brief "Recommendations" section suggesting modeling approaches
  based on what you found, including which features showed the most shift
  or the least importance (candidates to drop).


Overwriting submission/prompts/data_analyst.md


In [17]:
%%writefile submission/tools/data_analyst.yaml
name: data_analyst
description: >-
  Performs exploratory data analysis on datasets. Examines distributions,
  correlations, missing values, feature types, and potential data quality
  issues. Returns a structured analysis report.
model: gemini-3-flash-preview
instruction: !include ../prompts/data_analyst.md
tools:
  - run_command
  - write_file
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
  thinking_config:
    thinking_budget: 1024
    include_thoughts: true

Overwriting submission/tools/data_analyst.yaml


In [18]:
%%writefile submission/skills/feature-engineer/SKILL.md
---
name: feature-engineer
description: >-
  Provides a robust Python script for automated feature generation.
  Handles missing value imputation and numeric aggregations.
---

# Feature Engineer Skill

This skill equips the agent with a pre-packaged Python CLI script for automated feature engineering.

## Available Scripts

### 1. `generate_features.py`
Automatically identifies column types, imputes missing values, and calculates row mean.

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="feature_engineer",
    script_name="generate_features.py",
    args="--train train.csv --test test.csv --target target",
)
```
**Arguments**:
- `--train`: Path to training CSV (default: `train.csv`).
- `--test`: Path to test CSV (default: `test.csv`).
- `--target`: Name of the target column (default: `target`).

**Outputs**: Creates `train_engineered.csv` and `test_engineered.csv`.

---

## Domain Knowledge Resources

### `leakage_checklist.md`
A concise guide on preventing data leakage during feature engineering. You can read it using the `load_skill_resource` tool:
```python
load_skill_resource(
    skill_name="feature_engineer",
    resource_name="leakage_checklist.md",
)
```

Overwriting submission/skills/feature-engineer/SKILL.md


In [19]:
%%writefile submission/skills/feature-engineer/scripts/generate_features.py
#!/usr/bin/env python3
"""Robust CLI script for automated feature generation.

Identifies column types, imputes missing values, encodes categoricals,
adds row-wise aggregations, numeric interactions, and quantile binning.
All transforms are fit on train and applied to test to avoid leakage.
"""

import argparse
import os
import sys
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer


def detect_task_type(target_series):
    if target_series is None:
        return "regression"
    if target_series.dtype == object or target_series.nunique() <= 20:
        return "classification"
    return "regression"


def safe_step(name, fn):
    """Run a feature step, warn and continue on failure instead of crashing."""
    try:
        fn()
        print(f"  [ok] {name}")
    except Exception as exc:
        print(f"  [skip] {name} failed: {exc}")


def main():
    parser = argparse.ArgumentParser(description="Generate automated ML features.")
    parser.add_argument("--train", type=str, default="train.csv")
    parser.add_argument("--test", type=str, default="test.csv")
    parser.add_argument("--target", type=str, default="target")
    parser.add_argument("--task-type", type=str, default=None,
                         choices=[None, "classification", "regression"])
    parser.add_argument("--onehot-max-cardinality", type=int, default=10,
                         help="Categorical columns with <= this many unique "
                              "values are one-hot encoded; above it, "
                              "frequency-encoded instead.")
    args = parser.parse_args()

    print(f"Loading datasets: {args.train}, {args.test}...")
    for path in (args.train, args.test):
        if not os.path.exists(path):
            print(f"Error: file '{path}' not found.")
            sys.exit(1)

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    target_series = None
    if args.target in train_df.columns:
        target_series = train_df[args.target].copy()
        train_df = train_df.drop(columns=[args.target])
    else:
        print(f"Warning: target column '{args.target}' not found in train.")

    task_type = args.task_type or detect_task_type(target_series)
    print(f"Task type: {task_type}")

    # Align columns
    common_cols = [c for c in train_df.columns if c in test_df.columns]
    train_df = train_df[common_cols].copy()
    test_df = test_df[common_cols].copy()
    print(f"Initial shape: train={train_df.shape}, test={test_df.shape}")

    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

    # Missing-value count feature (computed before imputation)
    def add_missing_count():
        train_df["missing_count"] = train_df[common_cols].isna().sum(axis=1)
        test_df["missing_count"] = test_df[common_cols].isna().sum(axis=1)
    safe_step("missing_count feature", add_missing_count)

    # 1. Impute (fit on train, transform test)
    if num_cols:
        def impute_numeric():
            imputer = SimpleImputer(strategy="median")
            train_df[num_cols] = imputer.fit_transform(train_df[num_cols])
            test_df[num_cols] = imputer.transform(test_df[num_cols])
        safe_step(f"impute {len(num_cols)} numeric columns", impute_numeric)

    if cat_cols:
        def impute_categorical():
            imputer = SimpleImputer(strategy="most_frequent")
            train_df[cat_cols] = imputer.fit_transform(train_df[cat_cols])
            test_df[cat_cols] = imputer.transform(test_df[cat_cols])
        safe_step(f"impute {len(cat_cols)} categorical columns", impute_categorical)

    # 2. Row-wise numeric aggregations
    if num_cols:
        def add_aggregations():
            train_df["row_mean"] = train_df[num_cols].mean(axis=1)
            test_df["row_mean"] = test_df[num_cols].mean(axis=1)
            train_df["row_std"] = train_df[num_cols].std(axis=1)
            test_df["row_std"] = test_df[num_cols].std(axis=1)
            train_df["row_min"] = train_df[num_cols].min(axis=1)
            test_df["row_min"] = test_df[num_cols].min(axis=1)
            train_df["row_max"] = train_df[num_cols].max(axis=1)
            test_df["row_max"] = test_df[num_cols].max(axis=1)
        safe_step("row-wise mean/std/min/max", add_aggregations)

    # 3. Numeric interactions (top-variance columns only, to avoid feature blow-up)
    if len(num_cols) >= 2:
        def add_interactions():
            top_cols = train_df[num_cols].var().sort_values(ascending=False).index[:5].tolist()
            for i in range(len(top_cols)):
                for j in range(i + 1, len(top_cols)):
                    a, b = top_cols[i], top_cols[j]
                    train_df[f"{a}_x_{b}"] = train_df[a] * train_df[b]
                    test_df[f"{a}_x_{b}"] = test_df[a] * test_df[b]
                    denom_train = train_df[b].replace(0, np.nan)
                    denom_test = test_df[b].replace(0, np.nan)
                    train_df[f"{a}_div_{b}"] = (train_df[a] / denom_train).fillna(0)
                    test_df[f"{a}_div_{b}"] = (test_df[a] / denom_test).fillna(0)
        safe_step("top-variance numeric interactions", add_interactions)

    # 4. Quantile binning of skewed numeric features
    if num_cols:
        def add_bins():
            skewed = train_df[num_cols].skew().abs().sort_values(ascending=False)
            skewed_cols = skewed[skewed > 1.0].index[:5].tolist()
            for col in skewed_cols:
                bins = pd.qcut(train_df[col], q=5, duplicates="drop", retbins=True)[1]
                bins[0], bins[-1] = -np.inf, np.inf
                train_df[f"{col}_bin"] = pd.cut(train_df[col], bins=bins, labels=False)
                test_df[f"{col}_bin"] = pd.cut(test_df[col], bins=bins, labels=False)
        safe_step("quantile binning of skewed columns", add_bins)

    # 5. Categorical encoding (fit on train, applied to test)
    if cat_cols:
        def encode_categoricals():
            low_card = [c for c in cat_cols if train_df[c].nunique() <= args.onehot_max_cardinality]
            high_card = [c for c in cat_cols if c not in low_card]

            if low_card:
                combined = pd.concat([train_df[low_card], test_df[low_card]], axis=0)
                dummies = pd.get_dummies(combined, columns=low_card, dummy_na=False)
                train_dummies = dummies.iloc[:len(train_df)].reset_index(drop=True)
                test_dummies = dummies.iloc[len(train_df):].reset_index(drop=True)
                for c in low_card:
                    train_df.drop(columns=[c], inplace=True)
                    test_df.drop(columns=[c], inplace=True)
                for c in train_dummies.columns:
                    train_df[c] = train_dummies[c].values
                    test_df[c] = test_dummies[c].values if c in test_dummies.columns else 0

            for c in high_card:
                freq = train_df[c].value_counts(normalize=True)
                train_df[f"{c}_freq"] = train_df[c].map(freq).fillna(0)
                test_df[f"{c}_freq"] = test_df[c].map(freq).fillna(0)
                train_df.drop(columns=[c], inplace=True)
                test_df.drop(columns=[c], inplace=True)
        safe_step(f"encode {len(cat_cols)} categorical columns", encode_categoricals)

    # Re-attach target
    if target_series is not None:
        train_df[args.target] = target_series

    print(f"Engineered shape: train={train_df.shape}, test={test_df.shape}")
    train_df.to_csv("train_engineered.csv", index=False)
    test_df.to_csv("test_engineered.csv", index=False)
    print("Saved train_engineered.csv and test_engineered.csv successfully.")


if __name__ == "__main__":
    main()


Overwriting submission/skills/feature-engineer/scripts/generate_features.py


In [20]:
%%writefile submission/skills/feature-engineer/resources/leakage_checklist.md
# Data Leakage Prevention Checklist

Data leakage occurs when information from outside the training dataset is used to create the model, leading to overly optimistic performance estimates during local validation and catastrophic failure on the private leaderboard.

When performing feature engineering, strictly adhere to the following principles:

## Target Leakage Prevention
- **Rule**: Ensure no feature is directly derived from or highly correlated with the target column in a way that would not be available at true inference time.
- Watch for features that are proxies for the target (e.g., a column that's a rounded or shifted version of it).

## Fit-on-Train, Transform-on-Test
- **Rule**: Any statistic used for a transform (imputation medians, encoding frequencies, scaling parameters, quantile bin edges) must be computed on the training set only, then applied unchanged to the test set.
- Never fit an imputer, encoder, or scaler on train+test combined.

## Train/Test Distribution Shift
- **Rule**: If a feature's distribution differs substantially between train and test (check the `data_analyst`'s KS-test / PSI output), treat it with suspicion — it may not generalize, or may indicate a collection-process difference rather than a genuine signal.
- Consider dropping or down-weighting high-shift features if they don't clearly help CV score.

## Temporal Leakage
- **Rule**: If any column encodes time/order (dates, sequence IDs), never use future rows to construct features for past rows. Row-wise aggregations across a single row's own features are safe; aggregations that use "future" information relative to a row's timestamp are not.

## Aggregation Leakage
- **Rule**: Row-wise aggregations (mean/std/min/max across a single row's own features) are safe. Column-wise aggregations across the training set (e.g., group means, target encoding) must be fit per-CV-fold when used inside cross-validation, not on the full training set before splitting, or the CV score will be optimistic.

## Validation Discipline
- **Rule**: Use the same CV strategy you'll effectively be judged on. A mismatch between your local CV setup and the leaderboard's public/private split is a common source of surprises — prefer stratified folds for classification and check for any grouping structure that should keep related rows in the same fold.


Overwriting submission/skills/feature-engineer/resources/leakage_checklist.md


In [21]:
%%writefile submission/skills/model-trainer/SKILL.md
---
name: model-trainer
description: >-
  Trains and cross-validates baseline ML models (tree-based and linear),
  reports CV scores, and writes test-set predictions for each model.
---

# Model Trainer Skill

Runs k-fold cross-validation for a set of baseline models and saves
predictions, so the agent doesn't need to rewrite training/CV code from
scratch each iteration.

## Available Scripts

### 1. `train_models.py`

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="model-trainer",
    script_name="train_models.py",
    args="--train train_engineered.csv --test test_engineered.csv --target target --task-type classification --models rf,gbm,linear --cv 5",
)
```

**Arguments**:
- `--train` / `--test`: Paths to engineered CSVs.
- `--target`: Target column name.
- `--task-type`: `classification` or `regression` (auto-detected if omitted).
- `--models`: Comma-separated list from `rf`, `gbm`, `linear` (default: all three).
- `--cv`: Number of CV folds (default: 5).
- `--id-col`: ID column to carry through to prediction files (auto-detected
  from `id`/`Id`/`ID` if omitted).

**Outputs**:
- Prints per-model mean/std CV score to stdout.
- Writes `cv_scores.json` with all scores.
- Writes `preds_<model>.csv` (id + prediction) for each model, fit on full
  training data — ready to feed into the `ensembler` skill or
  `submit_predictions` directly.


Writing submission/skills/model-trainer/SKILL.md


In [22]:
%%writefile submission/skills/model-trainer/scripts/train_models.py
#!/usr/bin/env python3
"""Train and cross-validate baseline models, save CV scores and predictions."""

import argparse
import json
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error


def detect_task_type(y):
    if y.dtype == object or y.nunique() <= 20:
        return "classification"
    return "regression"


def get_model(name, task_type):
    if task_type == "classification":
        return {
            "rf": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
            "gbm": GradientBoostingClassifier(random_state=42),
            "linear": LogisticRegression(max_iter=1000, random_state=42),
        }[name]
    else:
        return {
            "rf": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
            "gbm": GradientBoostingRegressor(random_state=42),
            "linear": Ridge(random_state=42),
        }[name]


def score(y_true, y_pred, task_type):
    if task_type == "classification":
        try:
            return roc_auc_score(y_true, y_pred)
        except Exception:
            return accuracy_score(y_true, np.round(y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return -rmse  # higher (less negative) = better, kept consistent w/ "higher is better"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", default="train_engineered.csv")
    parser.add_argument("--test", default="test_engineered.csv")
    parser.add_argument("--target", default="target")
    parser.add_argument("--task-type", default=None, choices=[None, "classification", "regression"])
    parser.add_argument("--models", default="rf,gbm,linear")
    parser.add_argument("--cv", type=int, default=5)
    parser.add_argument("--id-col", default=None)
    args = parser.parse_args()

    for path in (args.train, args.test):
        if not os.path.exists(path):
            print(f"Error: file '{path}' not found.")
            sys.exit(1)

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    if args.target not in train_df.columns:
        print(f"Error: target '{args.target}' not in train.")
        sys.exit(1)

    y = train_df[args.target]
    task_type = args.task_type or detect_task_type(y)
    print(f"Task type: {task_type}")

    id_col = args.id_col
    if id_col is None:
        for cand in ("id", "Id", "ID"):
            if cand in test_df.columns:
                id_col = cand
                break

    feature_cols = [c for c in train_df.columns if c != args.target and c in test_df.columns]
    X = train_df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
    X_test = test_df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
    common = [c for c in X.columns if c in X_test.columns]
    X, X_test = X[common], X_test[common]

    y_enc = y.astype("category").cat.codes if y.dtype == object else y

    model_names = [m.strip() for m in args.models.split(",") if m.strip()]
    results = {}

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    splitter = (StratifiedKFold(n_splits=args.cv, shuffle=True, random_state=42)
                if task_type == "classification"
                else KFold(n_splits=args.cv, shuffle=True, random_state=42))

    for name in model_names:
        try:
            fold_scores = []
            use_scaled = (name == "linear")
            data_X = X_scaled if use_scaled else X
            split_arg = y_enc if task_type == "classification" else None
            for train_idx, val_idx in splitter.split(data_X, split_arg):
                model = get_model(name, task_type)
                model.fit(data_X.iloc[train_idx], y_enc.iloc[train_idx])
                if task_type == "classification" and hasattr(model, "predict_proba"):
                    val_pred = model.predict_proba(data_X.iloc[val_idx])[:, 1]
                else:
                    val_pred = model.predict(data_X.iloc[val_idx])
                fold_scores.append(score(y_enc.iloc[val_idx], val_pred, task_type))

            mean_score, std_score = float(np.mean(fold_scores)), float(np.std(fold_scores))
            results[name] = {"mean_cv_score": mean_score, "std_cv_score": std_score}
            print(f"{name}: mean_cv_score={mean_score:.5f} (+/- {std_score:.5f})")

            final_model = get_model(name, task_type)
            final_data_X = X_scaled if use_scaled else X
            final_test_X = X_test_scaled if use_scaled else X_test
            final_model.fit(final_data_X, y_enc)
            if task_type == "classification" and hasattr(final_model, "predict_proba"):
                test_pred = final_model.predict_proba(final_test_X)[:, 1]
            else:
                test_pred = final_model.predict(final_test_X)

            out = pd.DataFrame({args.target: test_pred})
            if id_col and id_col in test_df.columns:
                out.insert(0, id_col, test_df[id_col].values)
            out.to_csv(f"preds_{name}.csv", index=False)
            print(f"  saved preds_{name}.csv")
        except Exception as exc:
            print(f"[skip] model '{name}' failed: {exc}")

    with open("cv_scores.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Saved cv_scores.json")


if __name__ == "__main__":
    main()


Writing submission/skills/model-trainer/scripts/train_models.py


In [23]:
%%writefile submission/skills/model-trainer/scripts/train_models.py
#!/usr/bin/env python3
"""Train and cross-validate baseline models, save CV scores and predictions."""

import argparse
import json
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error


def detect_task_type(y):
    if y.dtype == object or y.nunique() <= 20:
        return "classification"
    return "regression"


def get_model(name, task_type):
    if task_type == "classification":
        return {
            "rf": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
            "gbm": GradientBoostingClassifier(random_state=42),
            "linear": LogisticRegression(max_iter=1000, random_state=42),
        }[name]
    else:
        return {
            "rf": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
            "gbm": GradientBoostingRegressor(random_state=42),
            "linear": Ridge(random_state=42),
        }[name]


def score(y_true, y_pred, task_type):
    if task_type == "classification":
        try:
            return roc_auc_score(y_true, y_pred)
        except Exception:
            return accuracy_score(y_true, np.round(y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return -rmse  # higher (less negative) = better, kept consistent w/ "higher is better"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", default="train_engineered.csv")
    parser.add_argument("--test", default="test_engineered.csv")
    parser.add_argument("--target", default="target")
    parser.add_argument("--task-type", default=None, choices=[None, "classification", "regression"])
    parser.add_argument("--models", default="rf,gbm,linear")
    parser.add_argument("--cv", type=int, default=5)
    parser.add_argument("--id-col", default=None)
    args = parser.parse_args()

    for path in (args.train, args.test):
        if not os.path.exists(path):
            print(f"Error: file '{path}' not found.")
            sys.exit(1)

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    if args.target not in train_df.columns:
        print(f"Error: target '{args.target}' not in train.")
        sys.exit(1)

    y = train_df[args.target]
    task_type = args.task_type or detect_task_type(y)
    print(f"Task type: {task_type}")

    id_col = args.id_col
    if id_col is None:
        for cand in ("id", "Id", "ID"):
            if cand in test_df.columns:
                id_col = cand
                break

    feature_cols = [c for c in train_df.columns if c != args.target and c in test_df.columns]
    X = train_df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
    X_test = test_df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
    common = [c for c in X.columns if c in X_test.columns]
    X, X_test = X[common], X_test[common]

    y_enc = y.astype("category").cat.codes if y.dtype == object else y

    model_names = [m.strip() for m in args.models.split(",") if m.strip()]
    results = {}

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    splitter = (StratifiedKFold(n_splits=args.cv, shuffle=True, random_state=42)
                if task_type == "classification"
                else KFold(n_splits=args.cv, shuffle=True, random_state=42))

    for name in model_names:
        try:
            fold_scores = []
            use_scaled = (name == "linear")
            data_X = X_scaled if use_scaled else X
            split_arg = y_enc if task_type == "classification" else None
            for train_idx, val_idx in splitter.split(data_X, split_arg):
                model = get_model(name, task_type)
                model.fit(data_X.iloc[train_idx], y_enc.iloc[train_idx])
                if task_type == "classification" and hasattr(model, "predict_proba"):
                    val_pred = model.predict_proba(data_X.iloc[val_idx])[:, 1]
                else:
                    val_pred = model.predict(data_X.iloc[val_idx])
                fold_scores.append(score(y_enc.iloc[val_idx], val_pred, task_type))

            mean_score, std_score = float(np.mean(fold_scores)), float(np.std(fold_scores))
            results[name] = {"mean_cv_score": mean_score, "std_cv_score": std_score}
            print(f"{name}: mean_cv_score={mean_score:.5f} (+/- {std_score:.5f})")

            final_model = get_model(name, task_type)
            final_data_X = X_scaled if use_scaled else X
            final_test_X = X_test_scaled if use_scaled else X_test
            final_model.fit(final_data_X, y_enc)
            if task_type == "classification" and hasattr(final_model, "predict_proba"):
                test_pred = final_model.predict_proba(final_test_X)[:, 1]
            else:
                test_pred = final_model.predict(final_test_X)

            out = pd.DataFrame({args.target: test_pred})
            if id_col and id_col in test_df.columns:
                out.insert(0, id_col, test_df[id_col].values)
            out.to_csv(f"preds_{name}.csv", index=False)
            print(f"  saved preds_{name}.csv")
        except Exception as exc:
            print(f"[skip] model '{name}' failed: {exc}")

    with open("cv_scores.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Saved cv_scores.json")


if __name__ == "__main__":
    main()


Overwriting submission/skills/model-trainer/scripts/train_models.py


In [24]:
%%writefile submission/skills/ensembler/SKILL.md
---
name: ensembler
description: >-
  Blends multiple prediction files (simple or CV-score-weighted averaging)
  into a single ensembled prediction, ready for submission.
---

# Ensembler Skill

## Available Scripts

### 1. `ensemble_predictions.py`

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="ensembler",
    script_name="ensemble_predictions.py",
    args="--preds preds_rf.csv,preds_gbm.csv,preds_linear.csv --target target --method weighted --scores cv_scores.json --output preds_ensemble.csv",
)
```

**Arguments**:
- `--preds`: Comma-separated list of prediction CSVs (same id/target column
  layout, e.g. from `model-trainer`).
- `--target`: Target/prediction column name to blend (default: `target`).
- `--method`: `mean` (simple average) or `weighted` (weight by CV score).
- `--scores`: Path to `cv_scores.json` (required for `weighted`; maps model
  name — inferred from filename `preds_<name>.csv` — to its mean CV score).
- `--output`: Output CSV path (default: `preds_ensemble.csv`).

**Note**: For classification, blend probabilities (not hard labels) for
best results — this is what `model-trainer`'s `preds_*.csv` files already
contain.


Writing submission/skills/ensembler/SKILL.md


In [25]:
%%writefile submission/skills/ensembler/scripts/ensemble_predictions.py
#!/usr/bin/env python3
"""Blend multiple prediction CSVs into a single ensembled prediction."""

import argparse
import json
import os
import sys
import numpy as np
import pandas as pd


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--preds", required=True, help="Comma-separated prediction CSV paths")
    parser.add_argument("--target", default="target")
    parser.add_argument("--method", default="mean", choices=["mean", "weighted"])
    parser.add_argument("--scores", default="cv_scores.json")
    parser.add_argument("--output", default="preds_ensemble.csv")
    args = parser.parse_args()

    paths = [p.strip() for p in args.preds.split(",") if p.strip()]
    if len(paths) < 2:
        print("Error: need at least 2 prediction files to ensemble.")
        sys.exit(1)

    dfs, names = [], []
    for p in paths:
        if not os.path.exists(p):
            print(f"Error: '{p}' not found.")
            sys.exit(1)
        dfs.append(pd.read_csv(p))
        base = os.path.basename(p)
        names.append(base.replace("preds_", "").replace(".csv", ""))

    id_col = None
    for cand in ("id", "Id", "ID"):
        if cand in dfs[0].columns:
            id_col = cand
            break

    n_rows = len(dfs[0])
    for i, df in enumerate(dfs):
        if len(df) != n_rows:
            print(f"Error: '{paths[i]}' has {len(df)} rows, expected {n_rows}.")
            sys.exit(1)
        if args.target not in df.columns:
            print(f"Error: '{paths[i]}' missing target column '{args.target}'.")
            sys.exit(1)

    weights = [1.0] * len(dfs)
    if args.method == "weighted":
        if not os.path.exists(args.scores):
            print(f"Error: scores file '{args.scores}' not found, needed for weighted method.")
            sys.exit(1)
        with open(args.scores) as f:
            score_map = json.load(f)
        raw_weights = []
        for name in names:
            if name in score_map:
                raw_weights.append(max(score_map[name]["mean_cv_score"], 1e-6))
            else:
                print(f"Warning: no CV score found for '{name}', using weight 1.0")
                raw_weights.append(1.0)
        total = sum(raw_weights)
        weights = [w / total for w in raw_weights]
        print(f"Weights: {dict(zip(names, weights))}")

    stacked = np.average(
        np.column_stack([df[args.target].values for df in dfs]),
        axis=1,
        weights=weights,
    )

    out = pd.DataFrame({args.target: stacked})
    if id_col:
        out.insert(0, id_col, dfs[0][id_col].values)
    out.to_csv(args.output, index=False)
    print(f"Saved ensembled predictions to {args.output} (blended {names})")


if __name__ == "__main__":
    main()


Writing submission/skills/ensembler/scripts/ensemble_predictions.py


In [26]:
%%writefile submission/skills/submission-validator/SKILL.md
---
name: submission-validator
description: >-
  Validates a prediction CSV's shape, columns, and value sanity before it is
  submitted, to avoid wasting a submission on a malformed file.
---

# Submission Validator Skill

## Available Scripts

### 1. `validate_submission.py`

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="submission-validator",
    script_name="validate_submission.py",
    args="--submission preds_ensemble.csv --test test.csv --target target",
)
```

**Arguments**:
- `--submission`: Path to the prediction CSV to validate.
- `--test`: Path to the original `test.csv` (used to check row count / id match).
- `--target`: Expected prediction column name.
- `--id-col`: ID column name (auto-detected from `id`/`Id`/`ID` if omitted).

**Behavior**: Prints a PASS/FAIL report and exits with code 1 on any
failure (wrong row count, missing columns, NaNs, id mismatch/duplicates,
constant predictions). Run this before every `submit_predictions` call.


Writing submission/skills/submission-validator/SKILL.md


In [27]:
%%writefile submission/skills/submission-validator/scripts/validate_submission.py
#!/usr/bin/env python3
"""Validate a submission CSV before it is submitted."""

import argparse
import os
import sys
import pandas as pd


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--submission", required=True)
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--target", default="target")
    parser.add_argument("--id-col", default=None)
    args = parser.parse_args()

    errors, warnings = [], []

    if not os.path.exists(args.submission):
        print(f"FAIL: submission file '{args.submission}' not found.")
        sys.exit(1)

    sub = pd.read_csv(args.submission)

    if args.target not in sub.columns:
        errors.append(f"missing target column '{args.target}'")

    id_col = args.id_col
    if id_col is None:
        for cand in ("id", "Id", "ID"):
            if cand in sub.columns:
                id_col = cand
                break

    if os.path.exists(args.test):
        test_df = pd.read_csv(args.test)
        if len(sub) != len(test_df):
            errors.append(f"row count mismatch: submission has {len(sub)}, test.csv has {len(test_df)}")
        if id_col and id_col in test_df.columns:
            if set(sub[id_col]) != set(test_df[id_col]):
                errors.append(f"id column '{id_col}' does not match test.csv ids exactly")
    else:
        warnings.append(f"'{args.test}' not found, skipping row-count/id checks")

    if args.target in sub.columns:
        if sub[args.target].isna().any():
            errors.append(f"target column has {sub[args.target].isna().sum()} NaN values")
        if sub[args.target].nunique() == 1:
            warnings.append("target column is constant across all rows - likely a bug")

    if id_col and sub[id_col].duplicated().any():
        errors.append(f"id column '{id_col}' has duplicate values")

    for w in warnings:
        print(f"WARN: {w}")

    if errors:
        for e in errors:
            print(f"FAIL: {e}")
        sys.exit(1)

    print(f"PASS: '{args.submission}' looks valid ({len(sub)} rows).")


if __name__ == "__main__":
    main()


Writing submission/skills/submission-validator/scripts/validate_submission.py


In [30]:
# Create the submission file
import subprocess

subprocess.run("cd submission && zip -r ../submission.zip . && cd -", shell=True, check=True)

from google.colab import files; files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>